In [1]:
# ======================================================================
# setup_data.ipynb
# FinGap-Governance | One-shot data setup
# ----------------------------------------------------------------------
# Unzips the three AI-Hub sample archives and sorts every file into the
# data/ layout the analysis notebooks (00-03) expect:
#
#   data/anomaly/raw/          <- 전자금융공동망 / 카드 transaction CSV
#   data/advisory/labeled/     <- 콜센터 상담 QAQ JSON  (_bk_/_in_/_se_)
#   data/accounting/labeled/   <- K-IFRS/일반/중소/특수 회계 QA JSON
#                                 (대분류/중분류 folder structure preserved)
#   data/cases/                <- (kept empty; research-authored walkthroughs)
#
# Rules:
#   * Only labelled data (02.라벨링데이터) is imported for anomaly/advisory.
#   * For accounting, the 대분류/중분류 sub-tree under 02.라벨링데이터
#     is preserved because 03_accounting_ifrs15.ipynb reads the standard
#     from the folder path.
#   * Raw (01.원천데이터) is skipped.
#   * Re-running is safe: the data/ tree is rebuilt from scratch.
#
# EDIT THESE TWO PATHS IF NEEDED:
UPLOADS_DIR = "."                       # folder containing the three zips
ZIPS = ["Sample.zip", "Sample (1).zip", "Sample (2).zip"]
# ======================================================================

import zipfile, shutil, re
from pathlib import Path

ROOT           = Path(".").resolve()
UPLOADS        = Path(UPLOADS_DIR).resolve()
DATA           = ROOT / "data"
ANOMALY_RAW    = DATA / "anomaly"    / "raw"
ADVISORY_LAB   = DATA / "advisory"   / "labeled"
ACCOUNTING_LAB = DATA / "accounting" / "labeled"
CASES          = DATA / "cases"

# ---- 0. Resolve the zip filenames (tolerate (1) vs __1_ variants) -----
def resolve_zip(name: str) -> Path | None:
    p = UPLOADS / name
    if p.exists():
        return p
    stem = re.sub(r"[ ()]+", "", Path(name).stem)          # "Sample (1)" -> "Sample1"
    for cand in UPLOADS.glob("*.zip"):
        if re.sub(r"[ ()_]+", "", cand.stem) == stem:
            return cand
    return None

zip_paths = []
for z in ZIPS:
    rp = resolve_zip(z)
    if rp is None:
        print(f"[warn] not found: {z}")
    else:
        zip_paths.append(rp)
assert zip_paths, f"No zips found in {UPLOADS}. Set UPLOADS_DIR correctly."

# ---- 1. Rebuild the data tree -----------------------------------------
if DATA.exists():
    shutil.rmtree(DATA)
for p in (ANOMALY_RAW, ADVISORY_LAB, ACCOUNTING_LAB, CASES):
    p.mkdir(parents=True, exist_ok=True)

# ---- 2. Classification rules ------------------------------------------
def classify(rel: str) -> str | None:
    low = rel.lower()
    if low.endswith(".csv") and ("전자금융" in rel or "카드" in rel or "efn" in low):
        return "anomaly"
    if "k-ifrs" in low or "회계기준" in rel or "회계처리" in rel:
        return "accounting"
    if re.search(r"_(bk|in|se)_\d", low) and low.endswith(".json"):
        return "advisory"
    if low.endswith(".csv"):
        return "anomaly"
    if low.endswith(".json"):
        return "advisory"
    return None

def is_labeled(rel: str) -> bool:
    return "02." in rel or "라벨링" in rel

# ---- 3. Extract + sort -------------------------------------------------
counts = {"anomaly": 0, "advisory": 0, "accounting": 0,
          "skipped_raw": 0, "unknown": 0}

for zp in zip_paths:
    with zipfile.ZipFile(zp) as zf:
        for info in zf.infolist():
            if info.is_dir():
                continue
            rel  = info.filename            # UTF-8 flagged; decodes cleanly
            task = classify(rel)
            if task is None:
                counts["unknown"] += 1
                continue
            # anomaly/advisory: labelled only
            if task in ("anomaly", "advisory") and not is_labeled(rel):
                counts["skipped_raw"] += 1
                continue

            name = Path(rel).name
            if task == "anomaly":
                dst = ANOMALY_RAW / name
            elif task == "advisory":
                dst = ADVISORY_LAB / name
            else:  # accounting: preserve 대분류/중분류 sub-tree
                parts = Path(rel).parts
                if "01.원천데이터" in parts:
                    counts["skipped_raw"] += 1
                    continue
                if "02.라벨링데이터" in parts:
                    idx = parts.index("02.라벨링데이터")
                    sub = Path(*parts[idx + 1:])
                else:
                    sub = Path(name)
                dst = ACCOUNTING_LAB / sub
                dst.parent.mkdir(parents=True, exist_ok=True)

            with zf.open(info) as src, open(dst, "wb") as out:
                shutil.copyfileobj(src, out)
            counts[task] += 1

# ---- 4. Report ---------------------------------------------------------
print("=" * 60)
print("Data setup complete.")
print(f"  anomaly   (CSV) : {counts['anomaly']:>4}  -> {ANOMALY_RAW}")
print(f"  advisory  (JSON): {counts['advisory']:>4}  -> {ADVISORY_LAB}")
print(f"  accounting(JSON): {counts['accounting']:>4}  -> {ACCOUNTING_LAB}")
print(f"  skipped (raw)   : {counts['skipped_raw']:>4}")
print(f"  unknown         : {counts['unknown']:>4}")
print("=" * 60)

# ---- 5. Sanity checks --------------------------------------------------
n_csv  = len(list(ANOMALY_RAW.glob("*.csv")))
n_adv  = len(list(ADVISORY_LAB.glob("*.json")))
n_acc  = len(list(ACCOUNTING_LAB.rglob("*.json")))
print(f"[check] anomaly csv files    : {n_csv}")
print(f"[check] advisory json files  : {n_adv}")
print(f"[check] accounting json files: {n_acc}")
print("[check] accounting standards found:",
      sorted({p.relative_to(ACCOUNTING_LAB).parts[0]
              for p in ACCOUNTING_LAB.rglob('*.json')}))
assert n_csv and n_adv and n_acc, "one or more task folders is empty — check zips"
print("\nOK — notebooks 00-03 can now run against data/.")

Data setup complete.
  anomaly   (CSV) :    1  -> C:\Users\User\Downloads\학술\50_fin_gap\notebooks\data\anomaly\raw
  advisory  (JSON):    5  -> C:\Users\User\Downloads\학술\50_fin_gap\notebooks\data\advisory\labeled
  accounting(JSON):  775  -> C:\Users\User\Downloads\학술\50_fin_gap\notebooks\data\accounting\labeled
  skipped (raw)   :  158
  unknown         :    0
[check] anomaly csv files    : 1
[check] advisory json files  : 5
[check] accounting json files: 775
[check] accounting standards found: ['K-IFRS', '일반기업회계기준', '중소기업회계기준', '특수및비영리조직회계기준']

OK — notebooks 00-03 can now run against data/.
